<a href="https://colab.research.google.com/github/JiaCheng0427/econ5200-lab01-data-portfolio/blob/main/lab_ch01_diagnostic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1: The Data Portfolio — The Economic Lens
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 40 min

---

**Format:** This lab contains **deliberately flawed code and analysis**. Your job:
1. Run the code
2. Identify what is wrong (not told what to look for)
3. Fix the issue
4. Document your reasoning
5. Extend the corrected analysis

**Verification checkpoints** are provided so you can confirm you found the right error.

---

In [1]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 1: Import libraries and load data
# -----------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

url = "https://raw.githubusercontent.com/TheEconomist/big-mac-data/master/output-data/big-mac-full-index.csv"
df = pd.read_csv(url, parse_dates=["date"])
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Countries: {df['name'].nunique()}, Periods: {df['date'].nunique()}")

Loaded: 2056 rows, 19 columns
Countries: 57, Periods: 45


<!-- 5200-onramp -->
## Part 0: Build It First (GUIDED — 25 min)

**Read this before anything else.** The rest of this lab hands you code that is
deliberately wrong and asks you to find the error. That is a professional
skill, and it is impossible to exercise against a baseline you have never
seen. So we build the correct version first.

This course assumes no prior programming. Part 0 of Labs 1 through 5 is where
that promise is kept — each one teaches the slice of Python that lab needs.
There is no separate primer to go and find; it is here, in the labs, next to
the data.

**This lab's slice:** the notebook itself, f-strings, the DataFrame, the
boolean mask, lists and `for`, `.groupby()`, dictionaries, and `def`. That is
everything Parts 1 to 3 ask you to read or write — nothing here is assumed.

### The notebook

You are in a **notebook**: a document of **cells**, each holding prose or code.
Run a code cell with **Shift+Enter**; output appears underneath.

- **Kernel** — the Python process behind the notebook. It remembers everything
  you have defined. Cells share one, so **order matters**: skip a cell and the
  ones after it may fail on a name that was never created.
- **Restart** — *Runtime → Restart session* empties it. When a notebook
  behaves impossibly, restart and run from the top.

One habit worth forming now: when a cell errors, read the **last** line of the
message first. That is the actual complaint; everything above it is the route
Python took to get there.

In [3]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 0a: variables, types, and f-strings
# -----------------------------------------------------------

# A VARIABLE is a name bound to a value with a single = sign.
# Read `x = 5` as "let x refer to 5", never as "x equals 5".
country   = "Switzerland"     # a str  — text, in quotes
big_mac   = 7.10              # a float — a number with a decimal point
n_periods = 45                # an int  — a whole number
print(type(country), type(big_mac), type(n_periods))

# An F-STRING is a string with `f` before the quote. Anything in { } is
# evaluated and dropped into the text. It is the most-used construct in this
# course, and the part after a COLON is a format spec controlling how a number
# is DISPLAYED — it never changes the underlying value.
#
#   :.1%    percent, 1 decimal        0.042 -> 4.2%
#   :.2f    fixed point, 2 decimals   7.1   -> 7.10
#   :,.0f   thousands separator       2056  -> 2,056
#   :>8     right-align in 8 columns
valuation = 0.4176
print(f"{country}: ${big_mac:.2f}, valued {valuation:.1%} against the US")

# :.1% MULTIPLIES BY 100 for you. This is the most common f-string bug:
print(f"WRONG: {valuation * 100:.1%}   <- 0.4176 * 100 = 41.76, shown as 4176.0%")
print(f"RIGHT: {valuation:.1%}")

<class 'str'> <class 'float'> <class 'int'>
Switzerland: $7.10, valued 41.8% against the US
WRONG: 4176.0%   <- 0.4176 * 100 = 41.76, shown as 4176.0%
RIGHT: 41.8%


In [4]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 0b: the DataFrame, one bracket vs two, and boolean masks
# -----------------------------------------------------------

# `df` was loaded by the Setup cell above. A DATAFRAME is a table: named
# columns, ordered rows. The INDEX is the row labels down the left — not a
# column. A DTYPE is the type of a whole column, and a surprising number of
# pandas errors are really a dtype that is not what you assumed.
print(f"Shape (rows, columns): {df.shape}")
print(f"Index: {df.index.min()} .. {df.index.max()}")
print(f"\nDtypes:\n{df[['name', 'date', 'local_price', 'dollar_ex']].dtypes}")

# ONE pair of brackets with one name gives a SERIES — a single column.
# TWO pairs gives a DATAFRAME, because the inner brackets are a LIST of names.
print(f"\ndf['name']          is a {type(df['name']).__name__}")
print(f"df[['name']]        is a {type(df[['name']]).__name__}")
print(f"df[['name','date']] is a {type(df[['name', 'date']]).__name__}")

# A BOOLEAN MASK is the single most important pattern in this course.
# A comparison on a column gives one True/False PER ROW; putting that inside
# df[ ... ] keeps the rows where it is True.
is_2024 = df["date"] == "2024-07-01"
print(f"\nThe mask is {len(is_2024)} True/False values; {is_2024.sum()} are True.")

# Combine with & (and), | (or), ~ (not) — NOT the words and/or/not, which
# raise "ValueError: The truth value of a Series is ambiguous".
# Each condition MUST have its own parentheses:
#     df[(df["a"] > 1) & (df["b"] < 2)]     correct
#     df[ df["a"] > 1  &  df["b"] < 2 ]     wrong — & binds tighter than >
expensive_2024 = df[(df["date"] == "2024-07-01") & (df["dollar_price"] > 6)]
print(f"\nBig Macs over $6 in July 2024: {len(expensive_2024)}")
print(expensive_2024[["name", "dollar_price"]].to_string(index=False))

Shape (rows, columns): (2056, 19)
Index: 0 .. 2055

Dtypes:
name                   object
date           datetime64[ns]
local_price           float64
dollar_ex             float64
dtype: object

df['name']          is a Series
df[['name']]        is a DataFrame
df[['name','date']] is a DataFrame

The mask is 2056 True/False values; 54 are True.

Big Macs over $6 in July 2024: 5
       name  dollar_price
  Argentina      6.545246
Switzerland      8.065890
  Euro area      6.059753
     Norway      6.767571
    Uruguay      7.071960


### Lists, loops, and the shape of a panel

Part 3 asks you to work out whether a table is cross-sectional, a time series
or a panel, and whether that panel is *balanced*. Both questions are counting
questions, and counting needs two things you have not met yet: a **list**, and
a **`for` loop** to walk one.


In [5]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 0c: lists, the for loop, and the shape of a panel
# -----------------------------------------------------------

# A LIST is an ordered box of values, in square brackets. Python counts from 0,
# and a slice x[1:4] includes 1 and EXCLUDES 4 — the right end always is.
key_columns = ["name", "date", "local_price", "dollar_ex", "dollar_price"]
print("How many:      ", len(key_columns))
print("First (idx 0): ", key_columns[0])
print("Last  (idx -1):", key_columns[-1])      # negative counts from the end
print("Slice [1:3]:   ", key_columns[1:3])     # positions 1,2 — NOT 3

# A FOR LOOP repeats a block once per element. The indentation is not
# decoration; it is how Python knows where the body ends.
print("\n--- dtype of each key column ---")
for col in key_columns:
    print(f"  {col:<14} {df[col].dtype}")

# .unique() lists the distinct values in a column; .nunique() just counts them.
# This is how you answer "how many units, how many periods" — which is exactly
# what tells cross-section from time series from panel.
n_countries = df["name"].nunique()
n_periods   = df["date"].nunique()
print(f"\nUnits (countries): {n_countries}")
print(f"Periods (dates):   {n_periods}")
print(f"Rows if every country appeared in every period: {n_countries * n_periods:,}")
print(f"Rows actually present:                          {len(df):,}")

# .groupby(col).size() splits the table into one group per distinct value and
# counts the rows in each. Split -> apply -> combine. Here it counts how many
# periods each country appears in — a PANEL BALANCE check.
periods_per_country = df.groupby("name").size()
print(f"\nperiods_per_country is a {type(periods_per_country).__name__}, "
      f"indexed by country:")
print(periods_per_country.head(3))

# The result is a Series, so a boolean mask works on it the same way it works
# on a column — the thing you compare and the thing you filter are the same.
complete = periods_per_country[periods_per_country == n_periods]
print(f"\nCountries present in all {n_periods} periods: {len(complete)} of {n_countries}")
print(f"This panel is {'BALANCED' if len(complete) == n_countries else 'UNBALANCED'}.")

How many:       5
First (idx 0):  name
Last  (idx -1): dollar_price
Slice [1:3]:    ['date', 'local_price']

--- dtype of each key column ---
  name           object
  date           datetime64[ns]
  local_price    float64
  dollar_ex      float64
  dollar_price   float64

Units (countries): 57
Periods (dates):   45
Rows if every country appeared in every period: 2,565
Rows actually present:                          2,056

periods_per_country is a Series, indexed by country:
name
Argentina     45
Australia     45
Azerbaijan    17
dtype: int64

Countries present in all 45 periods: 25 of 57
This panel is UNBALANCED.


### Dictionaries and functions

Part 3 asks you to write a function that hands back a **dictionary**. Those are
the last two pieces of the floor, and they arrive here rather than later
because this lab is the one that needs them.


In [6]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 0d: dictionaries, and measuring what is missing
# -----------------------------------------------------------

# A DICTIONARY is a lookup table: you get a value back by its KEY, not by its
# position. Written with { }, as key: value pairs. A list answers "what is
# third?"; a dict answers "what is the shape?".
profile = {
    "rows":      len(df),
    "columns":   df.shape[1],
    "countries": df["name"].nunique(),
}
print("The dict:      ", profile)
print("One value:     ", profile["countries"])
print("Its keys:      ", list(profile.keys()))

# You add a key by assigning to it. This is how a profile gets built up a piece
# at a time — and it is exactly the shape Part 3 asks you to return.
profile["periods"] = df["date"].nunique()
profile["balanced"] = len(df) == profile["countries"] * profile["periods"]
print("After adding two more:", profile)

# .isna() gives True/False per cell — a mask over the whole table. Because
# True counts as 1, .mean() on it is the SHARE missing, straight away.
print(f"\nShare of all cells that are missing: {df.isna().mean().mean():.1%}")

# Per column, sorted worst-first. .mean() on a DataFrame works column by column.
missing_pct = (df.isna().mean() * 100).round(1)
print("\nMissing % by column (worst five):")
print(missing_pct.sort_values(ascending=False).head(5).to_string())

# The same thing as a dict, built with a for loop over the columns. Read this
# closely — Part 3 asks you to produce it.
missing_by_col = {}
for col in df.columns:
    missing_by_col[col] = round(df[col].isna().mean() * 100, 1)

# Plain loop, not a one-liner: comprehensions arrive later, in the lab that
# needs them. Nothing here asks you to write anything you have not seen.
gappy = []
for col in df.columns:
    if missing_by_col[col] > 10:
        gappy.append(col)
print(f"\nColumns more than 10% empty ({len(gappy)}): {gappy}")

The dict:       {'rows': 2056, 'columns': 19, 'countries': 57}
One value:      57
Its keys:       ['rows', 'columns', 'countries']
After adding two more: {'rows': 2056, 'columns': 19, 'countries': 57, 'periods': 45, 'balanced': False}

Share of all cells that are missing: 4.6%

Missing % by column (worst five):
GBP_adjusted    12.4
CNY_adjusted    12.4
EUR_adjusted    12.4
adj_price       12.4
USD_adjusted    12.4

Columns more than 10% empty (7): ['GDP_bigmac', 'adj_price', 'USD_adjusted', 'EUR_adjusted', 'GBP_adjusted', 'JPY_adjusted', 'CNY_adjusted']


In [7]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 0e: def — packaging the work so it can be reused
# -----------------------------------------------------------

# A FUNCTION is a named recipe. `def` starts it, the indented block is the
# body, and `return` hands a value back to whoever called it. The names in the
# parentheses are PARAMETERS — placeholders filled in at the moment you call it.
#
# `= None` gives a parameter a DEFAULT, so the caller may leave it out.

def count_units(data, unit_col="name"):
    """Count the distinct units in `data`, looking at column `unit_col`."""
    return data[unit_col].nunique()


# Calling it. The value you pass takes the place of the parameter.
print("Distinct countries:", count_units(df))
print("Distinct currencies:", count_units(df, unit_col="currency_code"))

# A function that returns a DICT is the pattern Part 3 asks for: gather several
# facts, put each under a name, hand the whole thing back in one object.
def quick_profile(data, unit_col="name", time_col="date"):
    """Return a small structural profile of `data` as a dict."""
    out = {}
    out["shape"] = data.shape
    out["n_units"] = data[unit_col].nunique()
    out["n_periods"] = data[time_col].nunique()
    out["is_balanced"] = len(data) == out["n_units"] * out["n_periods"]
    return out


result = quick_profile(df)
print("\nquick_profile(df) returns a dict:")
for key in result:
    print(f"  {key:<13} {result[key]}")

# Because it takes the DataFrame as an argument, it works on any slice of it —
# which is the whole point of writing a function instead of a cell.
print("\nSame function, one cross-section:")
print(" ", quick_profile(df[df["date"] == "2024-07-01"]))

Distinct countries: 57
Distinct currencies: 57

quick_profile(df) returns a dict:
  shape         (2056, 19)
  n_units       57
  n_periods     45
  is_balanced   False

Same function, one cross-section:
  {'shape': (54, 19), 'n_units': 54, 'n_periods': 1, 'is_balanced': True}


In [9]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 0f: BUILD IT — the correct PPP valuation
# -----------------------------------------------------------
# This is the calculation Part 1 will hand you with an error in it. Build the
# right answer now, and keep the output on screen to compare against.
#
# The economics: implied PPP is the exchange rate that WOULD equalise Big Mac
# prices. Valuation asks how far the ACTUAL market rate sits from it.
#
#     implied_ppp   = local price / US price
#     valuation_pct = (implied_ppp - actual rate) / actual rate * 100
#
# Read the sign: implied ABOVE actual means the local currency buys less than
# PPP says it should -- it is OVERVALUED. Getting this ratio upside down is a
# real and common error, and it silently reverses every conclusion.

cs2024 = df[df["date"] == "2024-07-01"].copy()
us_price = cs2024.loc[cs2024["iso_a3"] == "USA", "dollar_price"].values[0]
print(f"US benchmark price: ${us_price:.2f}\n")

cs2024["implied_ppp"] = cs2024["local_price"] / us_price
cs2024["valuation_pct"] = (
    (cs2024["implied_ppp"] - cs2024["dollar_ex"]) / cs2024["dollar_ex"] * 100
)

print("Most OVERVALUED (correct):")
print(cs2024.nlargest(5, "valuation_pct")[["name", "valuation_pct"]]
      .to_string(index=False))
print("\nMost UNDERVALUED (correct):")
print(cs2024.nsmallest(5, "valuation_pct")[["name", "valuation_pct"]]
      .to_string(index=False))
print(f"\nMedian valuation: {cs2024['valuation_pct'].median():.1f}%")

US benchmark price: $5.69

Most OVERVALUED (correct):
       name  valuation_pct
Switzerland      41.755543
    Uruguay      24.287527
     Norway      18.937971
  Argentina      15.030678
  Euro area       6.498304

Most UNDERVALUED (correct):
        name  valuation_pct
      Taiwan     -59.899546
   Indonesia     -56.765824
       Egypt     -56.605698
       India     -54.031827
South Africa     -49.859051

Median valuation: -20.7%


### Now diagnose

Switzerland should head the overvalued list, with Taiwan and Indonesia at the
far end of the undervalued one. That is your baseline — **keep this output visible**.

Part 1 hands you the same calculation with one thing changed. Do not read the
code looking for a typo. Run it, look at the *answer*, and ask whether it can
be true. A currency that a moment ago was 60% cheap does not become the most
expensive in the world because of a rounding error.

---

## Part 1: Find the Bug — PPP Computation (10 min)

The following code computes Big Mac PPP valuations.
**Something is wrong with the formula.** Find it, fix it, explain.

In [18]:
# -----------------------------------------------------------
# GUIDED — Run as-is (contains deliberate error)
# Step 2: PPP computation — find the bug
# -----------------------------------------------------------

df_2024 = df[df["date"] == "2024-07-01"].copy()
us_price = df_2024.loc[df_2024["iso_a3"] == "USA", "dollar_price"].values[0]

# Compute implied PPP
df_2024["implied_ppp"] = df_2024["local_price"] / us_price

df_2024["valuation_pct"] = (df_2024["implied_ppp"]- df_2024["dollar_ex"])/df_2024["dollar_ex"] * 100

print("Top 5 'overvalued' currencies:")
print(df_2024.nlargest(5, "valuation_pct")[["name", "valuation_pct"]].to_string(index=False))
print()
print("Conclusion: Switzerland and Uruguay are the most OVERVALUED currencies in the world!")

Top 5 'overvalued' currencies:
       name  valuation_pct
Switzerland      41.755543
    Uruguay      24.287527
     Norway      18.937971
  Argentina      15.030678
  Euro area       6.498304

Conclusion: Switzerland and Uruguay are the most OVERVALUED currencies in the world!


### YOUR DIAGNOSIS

1. **What is wrong?** (identify the specific line and the mathematical error)
2. **Why does the bug produce backwards results?** (Indonesia and Egypt should be undervalued, not overvalued)
3. **Fix the code below** and report the correct top 5 overvalued countries

**Verification checkpoint:** After fixing, the July 2024 top five overvalued
should read Switzerland **+41.8%**, Uruguay +24.3%, Norway +18.9%,
Argentina +15.0%, Euro area +6.5%. Switzerland must come first; anything in
the +35% to +50% range is the right answer for it (the file this cell
downloads is The Economist's live repository, so the last decimal moves when
they revise a back-series). The five you saw before the fix — Indonesia,
Indonesia, Egypt, India, South Africa — should now be the five most
*under*valued, at -60% to -50%. If Indonesia is still in the top 5 overvalued,
you haven't found the bug.

1. What is wrong?
The valuation formula is incorrect. The correct formula should compare the implied purchasing power parity (PPP) with the actual exchange rate and use the actual exchange rate as the denominator.
2.Why does the bug produce backwards results? (Indonesia and Egypt should be undervalued, not overvalued)
because the incorrect valuation formula reverse the direction of comparison, so the currencies are actually undervalued appear to be o overvalued.
3.Fix the code below and report the correct top 5 overvalued countries
f_2024["valuation_pct"] = (df_2024["implied_ppp"]- df_2024["dollar_ex"])/df_2024["dollar_ex"] * 100
Top 5 'overvalued' currencies:
       name  valuation_pct
Switzerland      41.755543
    Uruguay      24.287527
     Norway      18.937971
  Argentina      15.030678
  Euro area       6.498304

In [9]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Fix the PPP computation
# -----------------------------------------------------------

# YOUR FIX HERE


## Part 2: Find the Methodological Flaw — Missing Data Handling (10 min)

---



The following analysis handles missing data before computing summary statistics.
The code runs correctly. The methodology is wrong. Find the flaw.

In [19]:
# -----------------------------------------------------------
# GUIDED — Run as-is (contains methodological flaw)
# Step 3: Missing data handling with flawed reasoning
# -----------------------------------------------------------

# Count observations per country
max_periods = df["date"].nunique()
country_counts = df.groupby("name")["date"].count()
incomplete = country_counts[country_counts < max_periods]

print(f"Countries with incomplete panels: {len(incomplete)}")
print(incomplete.sort_values())
print()

# "Solution": drop all countries with ANY missing periods
complete_countries = country_counts[country_counts == max_periods].index
df_clean = df[df["name"].isin(complete_countries)].copy()

print(f"Dropped {df['name'].nunique() - df_clean['name'].nunique()} countries")
print(f"Remaining: {df_clean['name'].nunique()} countries with complete panels")
print()

# Compute average dollar price over time
avg_price = df_clean.groupby("date")["dollar_price"].mean()
print("Average Big Mac price (complete-panel countries only):")
print(avg_price.tail())
print()
print("CONCLUSION: The global average Big Mac price has risen steadily.")
print("This represents the true trend for ALL countries worldwide.")

Countries with incomplete panels: 32
name
Azerbaijan              17
Bahrain                 17
Guatemala               17
Honduras                17
Lebanon                 17
Moldova                 17
Jordan                  17
Kuwait                  17
Nicaragua               17
Romania                 17
Qatar                   17
Oman                    17
United Arab Emirates    17
UAE                     22
Vietnam                 25
India                   31
Venezuela               31
Sri Lanka               35
Russia                  36
Israel                  36
Costa Rica              39
Ukraine                 39
Saudi Arabia            40
Colombia                40
Uruguay                 40
Pakistan                40
Norway                  41
Egypt                   42
Peru                    42
Turkey                  43
Denmark                 44
Philippines             44
Name: date, dtype: int64

Dropped 32 countries
Remaining: 25 countries with complete panels

A

### YOUR DIAGNOSIS

Three questions. Write your answers in the markdown cell below the next one.

1. **What is the methodological flaw?** The code runs and the arithmetic is
   right. The reasoning is wrong.
2. **Are the dropped countries a random subset?** Read the printed list before
   you characterise it.
3. **Which way is the "global average" biased, and by how much?** Measure it in
   the next cell rather than arguing about it.

**Checkpoint — 32 of 57 dropped, 25 kept.** Before writing "emerging markets get
dropped", check three names in the output above: **Argentina** is *kept* (all 45
periods), while **Norway** (41) and **Denmark** (44) are *dropped*. The filter
selects on continuity of measurement, not on development — and it conflates two
different things: countries that **left** the index (Venezuela 31, Russia 36,
Ukraine 39) and countries that **joined late** (the Gulf and Central American
block, 17 periods each).


1. What is the methodological flaw? The code runs and the arithmetic is right. The reasoning is wrong.
The methodological flaw is dropping every country that has any missing period. This creates selection bias because countries with incomplete data are excluded entirely. As a result, the calculated average only represents countries with complete panels, not all countries worldwide.
2. Are the dropped countries a random subset? Read the printed list before you characterise it.
No. The dropped countries are not a random subset. They are dropped because they do not have observations for all 45 periods. Some countries joined the index late, while others left the index earlier. Therefore, the filter selects countries based on continuity of data rather than randomly.
3. Which way is the "global average" biased, and by how much? Measure it in the next cell rather than arguing about it.
The complete-panel approach biases the global average upward. On average, it overstates the Big Mac price by about $0.081, or 2.1%. The complete-panel average is higher in 33 out of 45 periods.

In [22]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Demonstrate the bias
# Two blanks. Both use variables the cell above already made.
# -----------------------------------------------------------

# (a) the flawed approach — complete-panel countries only     [Step 0c]
#     `df_clean` above is already filtered to those 25.
complete_only = df_clean.groupby("date")["dollar_price"].mean()

# (b) the honest one — every country available in each period [Step 0c]
all_available = df.groupby("date")["dollar_price"].mean()

# The gap between the two IS the bias. Nothing below needs editing.
gap = complete_only - all_available
print(f"mean overstatement: ${gap.mean():+.3f}   ({(gap / all_available).mean():+.1%})")
print(f"complete-panel average is higher in {int((gap > 0).sum())} of {len(gap)} periods")
print("\nlast six periods      complete   all avail.      gap")
for d in complete_only.index[-6:]:
    print(f"  {str(d.date()):<16}{complete_only[d]:>9.3f}{all_available[d]:>12.3f}{gap[d]:>+9.3f}")

mean overstatement: $+0.081   (+2.1%)
complete-panel average is higher in 33 of 45 periods

last six periods      complete   all avail.      gap
  2024-01-01          4.456       4.381   +0.075
  2024-07-01          4.558       4.427   +0.131
  2025-01-01          4.527       4.434   +0.093
  2025-07-01          4.864       4.736   +0.128
  2026-01-01          5.058       4.900   +0.158
  2026-07-01          5.151       5.092   +0.059


## Part 3: Data Structure Profiling (YOUR TASK — 10 min)

One function, four blanks. Each is a single line, and the Step in brackets is
where you saw the move. You are not designing anything here — you are
assembling Part 0 into something reusable, which is what a `.py` module is.


In [25]:
# -----------------------------------------------------------
# ✏️ YOUR TASK — Profile the structure of any DataFrame
# Four blanks, one line each. Replace every ___
# -----------------------------------------------------------

def profile_dataframe(data, unit_col="name", time_col="date"):
    """Return a dict describing the structure and completeness of `data`."""
    profile = {}
    profile["shape"] = data.shape

    # 1. How many units, and how many periods?                    [Step 0c]
    profile["n_units"]   = data[unit_col].nunique()
    profile["n_periods"] = data[time_col].nunique()

    # The taxonomy follows from those two counts. Read this; it is the
    # whole point of the chapter.
    if profile["n_units"] > 1 and profile["n_periods"] > 1:
        profile["structure"] = "panel"
    elif profile["n_periods"] > 1:
        profile["structure"] = "time series"
    else:
        profile["structure"] = "cross-sectional"

    # 2. Count the periods each unit actually appears in.         [Step 0c]
    periods_per_unit = data.groupby(unit_col)[time_col].nunique()

    # 3. Keep only the units that appear in every period, and say whether
    #    that is all of them.                                     [Step 0c]
    complete = periods_per_unit[periods_per_unit == profile["n_periods"]]
    profile["complete_units"] = len(complete)
    profile["balanced"] = len(complete) == profile["n_units"]

    # 4. The share of each column that is missing, as a percentage.
    #    One entry per column, built with the loop.               [Step 0d]
    missing = {}
    for col in data.columns:
        missing[col] = data[col].isna().mean() * 100
    profile["missing"] = missing

    return profile


# --- run it on the Big Mac panel ---
result = profile_dataframe(df)
for key in result:
    if key != "missing":
        print(f"  {key:<16} {result[key]}")

n_gappy = 0
for col in result["missing"]:
    if result["missing"][col] > 10:
        n_gappy += 1
print(f"  {'cols >10% missing':<16} {n_gappy}")

  shape            (2056, 19)
  n_units          57
  n_periods        45
  structure        panel
  complete_units   25
  balanced         False
  cols >10% missing 7


---

### Where the module work went

An earlier version of this lab asked you to package `profile_dataframe()` and
two more functions into an importable `data_utils.py`, with type hints, full
docstrings and assertions.

That belongs in **Lab 2**, not here. Step 0e gave you enough of `def` to write
the function Part 3 asks for -- parameters, a default, a docstring, `return`.
Lab 2's Part 0 takes it further, to `raise` and the two routes a notebook uses
to write a real `.py` file, and its Part 4 builds one. Writing the module now
would put the packaging before the practice.

Keep the `profile_dataframe()` you wrote in Part 3. Lab 2 Part 4 picks it up.


---
## AI-Assisted Expansion: Comprehensive PPP Dashboard + Module

**The Generative AI Policy: Foundations First, Expansion Second.** You have now established manual mastery over PPP computation, missing data diagnosis, and data profiling. You are now authorized to operate under the "Co-Pilot Rule."

### Your Expansion Task (5200 — Advanced)
Build TWO artifacts:

**Artifact 1: `src/data_utils.py` module** with:
- `profile_dataframe(df, unit_col, time_col)` — comprehensive profiling
- `compute_valuation(df, benchmark)` — PPP computation pipeline
- `diagnose_missing(df, unit_col, time_col)` — missing data diagnosis with MCAR/MAR flags
- Full docstrings, type hints, and assertion-based validation

**Artifact 2: Interactive Streamlit app** that lets the user:
1. Upload any CSV or use the Big Mac Index
2. Auto-detect data structure (cross-sectional, time series, panel)
3. Profile missing data with interactive visualization
4. For panel data: toggle between balanced-only and all-available analyses
5. Show the bias quantification from dropping incomplete panels

### P.R.I.M.E. Prompt
Copy and paste this into Claude or ChatGPT:

In [26]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — Co-Pilot required
# -----------------------------------------------------------

# [Prep] Act as an expert Python Data Scientist specializing
# in data quality analysis and interactive dashboards.
#
# [Request] I just completed a diagnosis-first lab where I
# fixed a PPP computation bug, diagnosed survivorship bias
# from dropping incomplete panels, wrote a profile_dataframe()
# function, and measured the bias that dropping them introduces.
# Now I need TWO artifacts:
#
# 1. A reusable `src/data_utils.py` module with three functions:
#    - profile_dataframe(df, unit_col, time_col) -> dict
#    - compute_valuation(df, benchmark="USA") -> DataFrame
#    - diagnose_missing(df, unit_col, time_col) -> DataFrame
#    Include type hints, docstrings, and assertions.
#
# 2. A Streamlit app that profiles any uploaded CSV:
#    auto-detect structure, visualize missing data patterns,
#    show bias from dropping incomplete units.
#
# [Iterate] Use streamlit, pandas, plotly, scipy.stats.
# Use consistent variable names. No deprecated functions.
#
# [Mechanism Check] Add inline comments explaining:
#   - How auto-detection works (unit/time column heuristics)
#   - Why a complete-panel filter biases a global average
#   - How Streamlit session state handles file uploads
#
# [Evaluate] Explain what the dashboard reveals about data
# quality patterns and how this connects to Ch 6 (selection bias).

# PASTE AI-GENERATED CODE BELOW:


In [30]:
!mkdir -p src

In [31]:

%%writefile src/data_utils.py

from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency


def profile_dataframe(
    df: pd.DataFrame,
    unit_col: str = "name",
    time_col: str = "date",
) -> dict:
    """
    Profile the structure and completeness of a DataFrame.

    Returns information about shape, number of units and periods,
    panel structure, balance, and missing values.
    """
    assert isinstance(df, pd.DataFrame), "df must be a pandas DataFrame."
    assert not df.empty, "df cannot be empty."
    assert unit_col in df.columns, f"{unit_col} is not in the DataFrame."
    assert time_col in df.columns, f"{time_col} is not in the DataFrame."

    profile = {}

    profile["shape"] = df.shape
    profile["n_units"] = df[unit_col].nunique()
    profile["n_periods"] = df[time_col].nunique()

    if profile["n_units"] > 1 and profile["n_periods"] > 1:
        profile["structure"] = "panel"
    elif profile["n_periods"] > 1:
        profile["structure"] = "time series"
    else:
        profile["structure"] = "cross-sectional"

    periods_per_unit = (
        df.groupby(unit_col)[time_col].nunique()
    )

    complete_units = periods_per_unit[
        periods_per_unit == profile["n_periods"]
    ]

    profile["complete_units"] = len(complete_units)
    profile["balanced"] = (
        len(complete_units) == profile["n_units"]
    )

    profile["missing"] = (
        df.isna().mean() * 100
    ).round(1).to_dict()

    return profile


def compute_valuation(
    df: pd.DataFrame,
    benchmark: str = "USA",
) -> pd.DataFrame:
    """
    Compute implied PPP and currency valuation relative to a benchmark.

    Positive valuation_pct indicates overvaluation.
    Negative valuation_pct indicates undervaluation.
    """
    assert isinstance(df, pd.DataFrame), "df must be a DataFrame."

    required = {
        "date",
        "local_price",
        "dollar_ex",
        "dollar_price",
    }

    assert required.issubset(df.columns), (
        f"Missing required columns: {required - set(df.columns)}"
    )

    result = df.copy()

    if "iso_a3" in result.columns:
        benchmark_col = "iso_a3"
    elif "name" in result.columns:
        benchmark_col = "name"
    else:
        raise AssertionError(
            "Data must contain either iso_a3 or name."
        )

    benchmark_prices = (
        result.loc[
            result[benchmark_col] == benchmark,
            ["date", "dollar_price"],
        ]
        .drop_duplicates("date")
        .rename(columns={"dollar_price": "benchmark_price"})
    )

    assert not benchmark_prices.empty, (
        f"Benchmark '{benchmark}' was not found."
    )

    result = result.merge(
        benchmark_prices,
        on="date",
        how="left",
    )

    result["implied_ppp"] = (
        result["local_price"] / result["benchmark_price"]
    )

    result["valuation_pct"] = (
        (
            result["implied_ppp"]
            - result["dollar_ex"]
        )
        / result["dollar_ex"]
        * 100
    )

    return result


def diagnose_missing(
    df: pd.DataFrame,
    unit_col: str = "name",
    time_col: str = "date",
) -> pd.DataFrame:
    """
    Diagnose missing-data patterns by column.

    Chi-square tests check whether missingness is associated
    with units or time. These flags are diagnostics only;
    they do not prove that data are MCAR or MAR.
    """
    assert isinstance(df, pd.DataFrame), "df must be a DataFrame."
    assert unit_col in df.columns, f"{unit_col} is missing."
    assert time_col in df.columns, f"{time_col} is missing."

    rows = []

    for col in df.columns:
        missing = df[col].isna()

        missing_count = int(missing.sum())
        missing_pct = float(missing.mean() * 100)

        unit_p = np.nan
        time_p = np.nan

        if missing_count > 0 and missing.nunique() > 1:

            unit_table = pd.crosstab(
                df[unit_col],
                missing,
            )

            if (
                unit_table.shape[0] > 1
                and unit_table.shape[1] > 1
            ):
                _, unit_p, _, _ = chi2_contingency(
                    unit_table
                )

            time_table = pd.crosstab(
                df[time_col],
                missing,
            )

            if (
                time_table.shape[0] > 1
                and time_table.shape[1] > 1
            ):
                _, time_p, _, _ = chi2_contingency(
                    time_table
                )

        if missing_count == 0:
            flag = "No missing data"

        elif (
            (not pd.isna(unit_p) and unit_p < 0.05)
            or
            (not pd.isna(time_p) and time_p < 0.05)
        ):
            flag = "MAR-like pattern / not MCAR"

        else:
            flag = "MCAR plausible (not proven)"

        rows.append(
            {
                "column": col,
                "missing_count": missing_count,
                "missing_pct": round(missing_pct, 1),
                "unit_assoc_p": unit_p,
                "time_assoc_p": time_p,
                "missingness_flag": flag,
            }
        )

    return pd.DataFrame(rows)

Writing src/data_utils.py


---
## Digital Portfolio: Institutional Signaling

### Generate Your Professional README
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.**

In [ ]:
# -----------------------------------------------------------
# 🤖 AI EXPANSION — README generation (no code, just docs)
# -----------------------------------------------------------

# PASTE THIS PROMPT INTO CLAUDE:
#
# "I need help writing a project description for my data science lab.
# **Important Rule:** Do NOT generate any Python code for me.
#
# **What I did in this lab:**
# * Diagnosed and fixed a PPP computation bug (swapped numerator/denominator)
# * Identified survivorship bias from dropping incomplete panels
# * Quantified the bias: complete-panel countries had $X.XX higher average
#   Big Mac prices than incomplete-panel countries. Report the difference AND
#   how large it is in dollars and as a share -- report what your own
#   single July 2024 cross-section the Welch test does NOT reach significance
#   (n = 25 vs 29, skewed prices); the evidence for the bias is that the
#   period-by-period panel comparison shows the same sign in every period.
#   Write what your own output says, not what you expected it to say.
# * Built a reusable data_utils.py module with profiling, PPP computation,
#   and missing data diagnosis functions
# * Created a Streamlit dashboard for automated data quality profiling
#
# **Please write a README.md entry including:**
# 1. Project Title: Data Quality Profiling — Big Mac Index
# 2. Objective: A professional one-sentence summary
# 3. Methodology: Bullet points of technical steps
# 4. Key Findings: Summary of results
# Make this sound like a professional tech economist wrote it."

In [33]:
%%writefile app.py

from __future__ import annotations

from io import BytesIO

import pandas as pd
import plotly.express as px
import streamlit as st

from src.data_utils import (
    diagnose_missing,
    profile_dataframe,
)


BIG_MAC_URL = (
    "https://raw.githubusercontent.com/"
    "TheEconomist/big-mac-data/"
    "master/output-data/big-mac-full-index.csv"
)


def guess_time_column(df: pd.DataFrame) -> str:
    """
    Guess which column represents time.
    """

    preferred_names = [
        "date",
        "time",
        "year",
        "period",
        "month",
        "quarter",
    ]

    lower_names = {
        str(col).lower(): col
        for col in df.columns
    }

    # Auto-detection first checks common time-column names.
    for name in preferred_names:
        if name in lower_names:
            return lower_names[name]

    # If no obvious name exists, choose the column that
    # can most successfully be interpreted as dates.
    best_col = df.columns[0]
    best_score = -1.0

    for col in df.columns:
        parsed = pd.to_datetime(
            df[col],
            errors="coerce",
        )

        score = parsed.notna().mean()

        if score > best_score:
            best_score = score
            best_col = col

    return best_col


def guess_unit_column(
    df: pd.DataFrame,
    time_col: str,
) -> str:
    """
    Guess which column identifies units such as countries.
    """

    preferred_names = [
        "name",
        "country",
        "entity",
        "unit",
        "iso_a3",
        "id",
    ]

    lower_names = {
        str(col).lower(): col
        for col in df.columns
    }

    for name in preferred_names:
        if (
            name in lower_names
            and lower_names[name] != time_col
        ):
            return lower_names[name]

    candidates = [
        col
        for col in df.columns
        if col != time_col
    ]

    # Repeated values are useful for identifying panel units.
    for col in candidates:
        n_unique = df[col].nunique()

        if 1 < n_unique < len(df):
            return col

    return candidates[0]


st.set_page_config(
    page_title="Data Quality Profiling",
    layout="wide",
)

st.title("Data Quality Profiling — Big Mac Index")

st.write(
    """
    This dashboard profiles a dataset's structure,
    examines missing-data patterns, and measures the
    bias introduced by dropping incomplete panel units.
    """
)


# ---------------------------------------------------------
# DATA SOURCE
# ---------------------------------------------------------

source = st.sidebar.radio(
    "Choose a data source",
    [
        "Big Mac Index",
        "Upload CSV",
    ],
)


if source == "Upload CSV":

    uploaded_file = st.sidebar.file_uploader(
        "Upload a CSV file",
        type=["csv"],
    )

    if uploaded_file is None:
        st.info("Upload a CSV file to begin.")
        st.stop()

    file_bytes = uploaded_file.getvalue()

    file_key = (
        uploaded_file.name,
        len(file_bytes),
        hash(file_bytes),
    )

    # Streamlit reruns the script whenever a widget changes.
    # session_state keeps the uploaded data available between
    # those reruns instead of reading it from scratch each time.
    if st.session_state.get("file_key") != file_key:

        st.session_state["file_key"] = file_key

        st.session_state["uploaded_df"] = pd.read_csv(
            BytesIO(file_bytes)
        )

    df = st.session_state["uploaded_df"].copy()


else:

    @st.cache_data
    def load_big_mac() -> pd.DataFrame:
        return pd.read_csv(
            BIG_MAC_URL,
            parse_dates=["date"],
        )

    df = load_big_mac()


# ---------------------------------------------------------
# PREVIEW
# ---------------------------------------------------------

st.subheader("1. Data Preview")

st.dataframe(
    df.head(20),
    width="stretch",
)


# ---------------------------------------------------------
# AUTO-DETECT STRUCTURE
# ---------------------------------------------------------

auto_time = guess_time_column(df)

time_col = st.sidebar.selectbox(
    "Time column",
    options=list(df.columns),
    index=list(df.columns).index(auto_time),
)

auto_unit = guess_unit_column(
    df,
    time_col,
)

unit_col = st.sidebar.selectbox(
    "Unit column",
    options=list(df.columns),
    index=list(df.columns).index(auto_unit),
)


profile = profile_dataframe(
    df,
    unit_col=unit_col,
    time_col=time_col,
)


st.subheader("2. Data Structure")

col1, col2, col3, col4 = st.columns(4)

col1.metric(
    "Rows",
    profile["shape"][0],
)

col2.metric(
    "Units",
    profile["n_units"],
)

col3.metric(
    "Periods",
    profile["n_periods"],
)

col4.metric(
    "Structure",
    profile["structure"],
)


col5, col6 = st.columns(2)

col5.metric(
    "Complete units",
    profile["complete_units"],
)

col6.metric(
    "Balanced panel?",
    str(profile["balanced"]),
)


# ---------------------------------------------------------
# MISSING DATA
# ---------------------------------------------------------

st.subheader("3. Missing Data")

missing_df = diagnose_missing(
    df,
    unit_col=unit_col,
    time_col=time_col,
)

st.dataframe(
    missing_df,
    width="stretch",
)


missing_bar = px.bar(
    missing_df.sort_values(
        "missing_pct",
        ascending=False,
    ),
    x="column",
    y="missing_pct",
    title="Missing Values by Column (%)",
)

st.plotly_chart(
    missing_bar,
    width="stretch",
)


missing_matrix = (
    df.head(200)
    .isna()
    .astype(int)
    .T
)

missing_heatmap = px.imshow(
    missing_matrix,
    aspect="auto",
    title="Missing-Data Pattern",
)

st.plotly_chart(
    missing_heatmap,
    width="stretch",
)


# ---------------------------------------------------------
# PANEL BIAS
# ---------------------------------------------------------

if profile["structure"] == "panel":

    st.subheader(
        "4. Complete-Panel vs All-Available Analysis"
    )

    numeric_cols = list(
        df.select_dtypes(
            include="number"
        ).columns
    )

    if not numeric_cols:

        st.warning(
            "No numeric variable is available for comparison."
        )

    else:

        if "dollar_price" in numeric_cols:
            default_variable = "dollar_price"
        else:
            default_variable = numeric_cols[0]

        value_col = st.selectbox(
            "Variable to average",
            options=numeric_cols,
            index=numeric_cols.index(
                default_variable
            ),
        )

        display_mode = st.radio(
            "Display",
            [
                "Compare both",
                "All available",
                "Complete-panel only",
            ],
            horizontal=True,
        )

        n_periods = df[time_col].nunique()

        periods_per_unit = (
            df.groupby(unit_col)[time_col]
            .nunique()
        )

        complete_units = (
            periods_per_unit[
                periods_per_unit == n_periods
            ]
            .index
        )

        complete_df = df[
            df[unit_col].isin(complete_units)
        ].copy()

        complete_only = (
            complete_df
            .groupby(time_col)[value_col]
            .mean()
        )

        all_available = (
            df
            .groupby(time_col)[value_col]
            .mean()
        )

        comparison = pd.concat(
            [
                complete_only.rename(
                    "complete_only"
                ),
                all_available.rename(
                    "all_available"
                ),
            ],
            axis=1,
        ).dropna()

        comparison["gap"] = (
            comparison["complete_only"]
            - comparison["all_available"]
        )

        comparison["gap_pct"] = (
            comparison["gap"]
            / comparison["all_available"]
            * 100
        )

        # The complete-panel filter is not random.
        # Units that joined late or left early are removed.
        # Because sample membership depends on continuity
        # of observation, the resulting global average can
        # differ systematically from the all-available sample.

        mean_gap = comparison["gap"].mean()

        mean_gap_pct = (
            comparison["gap_pct"].mean()
        )

        higher_periods = int(
            (comparison["gap"] > 0).sum()
        )

        metric1, metric2, metric3 = st.columns(3)

        metric1.metric(
            "Mean bias",
            f"${mean_gap:+.3f}",
        )

        metric2.metric(
            "Mean bias (%)",
            f"{mean_gap_pct:+.1f}%",
        )

        metric3.metric(
            "Complete-panel higher",
            f"{higher_periods} of {len(comparison)} periods",
        )


        if display_mode == "All available":

            plot_df = comparison[
                ["all_available"]
            ]

        elif display_mode == "Complete-panel only":

            plot_df = comparison[
                ["complete_only"]
            ]

        else:

            plot_df = comparison[
                [
                    "complete_only",
                    "all_available",
                ]
            ]


        plot_df = (
            plot_df
            .reset_index()
            .melt(
                id_vars=[time_col],
                var_name="sample",
                value_name=value_col,
            )
        )

        comparison_plot = px.line(
            plot_df,
            x=time_col,
            y=value_col,
            color="sample",
            markers=True,
            title=(
                "Complete-Panel vs "
                "All-Available Average"
            ),
        )

        st.plotly_chart(
            comparison_plot,
            width="stretch",
        )


# ---------------------------------------------------------
# EVALUATION
# ---------------------------------------------------------

st.subheader("5. Evaluation")

st.markdown(
    """
The dashboard shows that data quality is not simply a
data-cleaning issue; it can change an economic conclusion.

Missing observations may be concentrated in particular
countries or time periods. Dropping every unit with an
incomplete history therefore changes the composition of the
sample rather than removing observations randomly.

This is a form of **selection bias**. The arithmetic for the
complete-panel sample may be correct, but the sample itself
has been selected according to continuity of observation.

Comparing complete-panel and all-available averages makes
the direction and magnitude of that bias visible instead of
assuming that the restricted sample represents the full
population.
"""
)


Overwriting app.py


In [35]:
!python -m py_compile src/data_utils.py app.py
from src.data_utils import profile_dataframe, compute_valuation, diagnose_missing

print("Imports successful")

Imports successful


---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab01-data-portfolio`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab01-data-portfolio`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab01-data-portfolio`, branch `main`,
   commit message `Lab 01 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the P.R.I.M.E. prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. The `.py` module this lab asks you to write is **not** pushed by step 2 —
   that step sends the notebook only. On the repo page use **Add file →
   Upload files**, drag the `.py` in, and **Commit changes**. It is part of
   what is graded.
5. On Canvas: upload the `.ipynb` **and** paste your repository URL.

No terminal, no token, nothing to install.
